# Handwritten Digit Recognition

**Goal:** look at an 8x8 picture of a handwritten digit and predict which digit it is (0-9).

**What you will learn:**
- How image data is stored as numbers (a grid of pixel values).
- How to display an image with matplotlib.
- How to train a Support Vector Machine (SVM) classifier.
- How to compare a couple of SVM settings.

**Problem type:** classification with 10 classes (multi-class).

**Dataset:** `data/digits.csv` - 1,797 images. Columns `pixel_0` ... `pixel_63` are the 64 pixel brightness values (0-16) of an 8x8 image, and `label` is the true digit.

## Import the libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn import svm
from sklearn.metrics import accuracy_score

## Load the dataset

The data lives in the `data/` folder, one level up from this notebook. The helper below resolves that relative path whether you launch the notebook from the project root or from inside `notebooks/`.

In [ ]:
def find_data(filename):
    for base in [".", "data", "../data", "../../data"]:
        path = os.path.join(base, filename)
        if os.path.exists(path):
            return path
    raise FileNotFoundError(
        f"Could not find {filename}. Keep the data/ folder next to the notebooks/ folder."
    )


DATA_PATH = find_data("digits.csv")
df = pd.read_csv(DATA_PATH)

print("Loaded:", DATA_PATH)
print("Rows, columns:", df.shape)
df.head()

## Look at an image

Each row is one image flattened into 64 numbers. Reshaping to 8x8 turns it back into a picture we can see.

In [ ]:
pixel_columns = [col for col in df.columns if col.startswith("pixel_")]

image = df.loc[0, pixel_columns].values.reshape(8, 8)

print("This image shows the digit:", df.loc[0, "label"])
plt.imshow(image, cmap="gray")
plt.title(f"Label = {df.loc[0, 'label']}")
plt.axis("off")
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.ravel()):
    img = df.loc[i, pixel_columns].values.reshape(8, 8)
    ax.imshow(img, cmap="gray")
    ax.set_title(f"label={df.loc[i, 'label']}")
    ax.axis("off")
plt.tight_layout()
plt.show()

## Split features and target

In [ ]:
X = df[pixel_columns].values
y = df["label"].values

print("X shape:", X.shape)
print("y shape:", y.shape)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print("Training rows:", X_train.shape[0])
print("Test rows:    ", X_test.shape[0])

## Train a Support Vector Machine

An SVM finds the boundary that best separates the classes. We start with a straight-line (`linear`) kernel.

In [ ]:
model = svm.SVC(kernel="linear")
model.fit(X_train, y_train)

predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)
print(f"Accuracy: {accuracy:.2%}")

## Check a single prediction

`predict()` expects a 2D array (a list of rows), so we reshape one image to `(1, 64)`.

In [ ]:
n = 15
single_image = df.loc[n, pixel_columns].values.reshape(1, -1)
predicted = model.predict(single_image)[0]
actual = df.loc[n, "label"]

plt.imshow(df.loc[n, pixel_columns].values.reshape(8, 8), cmap="gray")
plt.title(f"Predicted = {predicted} | Actual = {actual}")
plt.axis("off")
plt.show()

## Compare a few SVM settings

The `kernel` and `gamma` settings change how flexible the boundary is. Let us try four combinations.

In [ ]:
models = {
    "linear": svm.SVC(kernel="linear"),
    "rbf": svm.SVC(kernel="rbf"),
    "rbf (gamma=0.001)": svm.SVC(gamma=0.001),
    "rbf (gamma=0.001, C=0.1)": svm.SVC(gamma=0.001, C=0.1),
}

for name, m in models.items():
    m.fit(X_train, y_train)
    acc = accuracy_score(y_test, m.predict(X_test))
    print(f"{name:28s} accuracy: {acc:.2%}")

## What you learned

- Images are just numbers - an 8x8 grid of pixel values.
- The same `fit` / `predict` / `accuracy_score` workflow works for 10 classes as easily as for 2.
- Changing model settings (kernel, gamma, C) changes the result - always compare on a test set.

**Try next:** look at the rows the model got wrong and plot a few of them. Are the mistakes understandable?